### Solving the Optimization Problem with Julia: adding sites to the existing network

Mixed-integer facility location model (p-median) for choosing **P = 5** new ambulance standby sites, **given the existing FDNY firehouses**.

Each ZIP area is served by whichever site reaches it fastest: either the nearest existing firehouse (anywhere in Brooklyn), with time $s_i$, or one of the newly opened candidate sites (anywhere in Brooklyn). A new site only improves a ZIP's time where it beats every existing firehouse.

$$
\min_{x,y,z}\ \frac{\sum_{i\in I} w_i\left(s_i\, z_i + \sum_{j\in J} t_{ji}\, x_{ji}\right)}{\sum_{i\in I} w_i}
\quad \text{s.t.}\quad \sum_{j\in J} y_j = P,\qquad z_i + \sum_{j\in J} x_{ji} = 1\ \ (\forall i\in I),\qquad x_{ji}\le y_j\ \ (\forall i\in I,\ j\in J),\qquad x,y,z\in\{0,1\}
$$

- $I$: ZIP areas (demand points), $w_i$: call count in area $i$ during the planning period
- $J$: candidate standby sites, $t_{ji}$: estimated driving time (minutes) from candidate $j$ to area $i$'s representative point
- $K$: existing firehouses, $s_i = \min_{k\in K} t_{ki}$: driving time from the nearest existing firehouse to area $i$
- $y_j$: candidate $j$ is opened; $x_{ji}$: candidate $j$ serves area $i$; $z_i$: area $i$ stays with its nearest existing firehouse

Existing firehouses are always open, so they need no decision variables; only their best time per ZIP, $s_i$, enters the model. With $P = 0$ the objective equals the status-quo average.

Data files:
- `data/demand.csv`: one row per Brooklyn ZIP area: `area_id, lat, lon, calls` (built in `Analysis.ipynb` / `Geography.ipynb`; ZIP 11385 already excluded).
- `data/candidate_sites.csv`: the 20 proposed standby points $J$ (`site_id, lat, lon, source`), screened with `screen_candidate_sites.py`.
- `data/travel_times_routed.csv`: $t_{ji}$ for the candidates, long format `site_id, area_id, ..., minutes`. Built by `TravelTimes_Routed.ipynb`.
- `data/FDNY_Firehouse_Listing_20261115.csv`: existing firehouses $K$. The Brooklyn rows are numbered `FH_001`–`FH_066` in file order, the same rule `TravelTimes.ipynb` uses.
- `data/travel_times.csv`: $t_{ki}$ for the existing firehouses. Built by `TravelTimes.ipynb`.

**Caveat:** both travel-time files use 4 min/mile, but they come from different routing pipelines. For a like-for-like comparison of $s_i$ and $t_{ji}$, route the firehouses through `TravelTimes_Routed.ipynb` as well, save them to a separate file, and point `EXISTING_TIME_FILE` at it.

In [1]:
using CSV, DataFrames, JuMP, Gurobi, Printf

#### 1. Configuration

In [2]:
DATA_DIR   = "data"
OUTPUT_DIR = "results"

DEMAND_FILE        = joinpath(DATA_DIR, "demand.csv")                     # area_id, lat, lon, calls
CANDIDATE_FILE     = joinpath(DATA_DIR, "candidate_sites.csv")            # site_id, lat, lon, source (20 pins)
TRAVEL_TIME_FILE   = joinpath(DATA_DIR, "travel_times_routed.csv")        # site_id, area_id, ..., minutes
EXISTING_FILE      = joinpath(DATA_DIR, "FDNY_Firehouse_Listing_20261115.csv")  # all NYC firehouses
BOROUGH            = "Brooklyn"
EXISTING_TIME_FILE = joinpath(DATA_DIR, "travel_times.csv")               # site_id, area_id, ..., minutes

P          = 5               # number of new standby sites to open
WEIGHT_COL = :calls          # w_i: call count per ZIP area
TIME_COL   = :minutes        # time column in both travel-time files

TIME_LIMIT_SEC = 300

300

#### 2. Load demand areas ($I$, $w_i$)

In [3]:
demand = CSV.read(DEMAND_FILE, DataFrame)
demand.area_id = string.(demand.area_id)
dropmissing!(demand, [:lat, :lon, WEIGHT_COL])
filter!(row -> row[WEIGHT_COL] > 0, demand)

I_areas = demand.area_id
w = Float64.(demand[!, WEIGHT_COL])
println("Demand areas |I| = $(length(I_areas)),  total $(WEIGHT_COL) = $(sum(w))")
first(demand, 5)

Demand areas |I| = 37,  total calls = 202491.0


Row,area_id,lat,lon,calls
,String,Float64,Float64,Int64
1,11201,40.6946,-73.9903,11460
2,11203,40.6487,-73.9339,7205
3,11204,40.621,-73.9859,2877
4,11205,40.6965,-73.9664,3461
5,11206,40.7024,-73.9419,8664


#### 3. Load candidate standby sites ($J$) and existing firehouses ($K$)

In [4]:
candidates = CSV.read(CANDIDATE_FILE, DataFrame)
candidates.site_id = string.(candidates.site_id)
J_sites = candidates.site_id

# Same construction as TravelTimes.ipynb, so site_ids line up with EXISTING_TIME_FILE.
firehouses = CSV.read(EXISTING_FILE, DataFrame; normalizenames=true)
firehouses = dropmissing(filter(r -> r.Borough == BOROUGH, firehouses), [:Latitude, :Longitude])
existing = DataFrame(
    site_id = [@sprintf("FH_%03d", k) for k in 1:nrow(firehouses)],
    name    = firehouses.FacilityName,
    address = firehouses.FacilityAddress,
    lat     = Float64.(firehouses.Latitude),
    lon     = Float64.(firehouses.Longitude),
)
K_sites = existing.site_id

isempty(intersect(J_sites, K_sites)) || error("Candidate and existing site_ids overlap")
println("Candidate sites |J| = $(length(J_sites)),  existing firehouses |K| = $(length(K_sites))")
first(candidates, 5)

Candidate sites |J| = 20,  existing firehouses |K| = 66


Row,site_id,lat,lon,source
,String,Float64,Float64,String15
1,1,40.6382,-73.9125,alternate
2,2,40.6611,-73.8861,alternate
3,3,40.6547,-73.9392,original
4,4,40.6217,-73.9494,original
5,5,40.5901,-73.941,alternate


#### 4. Travel times

`t[j, i]` = minutes from candidate `j` to area `i`; `t_exist[k, i]` = minutes from existing firehouse `k` to area `i`. Every site can serve every ZIP, regardless of which ZIP it sits in. `s[i]` is the time from the nearest existing firehouse.

In [5]:
function load_travel_times(path, sites, I_areas; col=TIME_COL)
    tt = CSV.read(path, DataFrame)
    tt.site_id = string.(tt.site_id)
    tt.area_id = string.(tt.area_id)
    jidx = Dict(s => k for (k, s) in enumerate(sites))
    iidx = Dict(a => k for (k, a) in enumerate(I_areas))
    t = fill(NaN, length(sites), length(I_areas))
    for r in eachrow(tt)
        haskey(jidx, r.site_id) && haskey(iidx, r.area_id) || continue
        t[jidx[r.site_id], iidx[r.area_id]] = r[col]
    end
    n_missing = count(isnan, t)
    n_missing == 0 || error("$(n_missing) (site, area) pairs missing from $(path): check that the site file and travel-time file use the same site_ids")
    return t
end

t       = load_travel_times(TRAVEL_TIME_FILE, J_sites, I_areas)
t_exist = load_travel_times(EXISTING_TIME_FILE, K_sites, I_areas)

s = vec(minimum(t_exist; dims=1))                          # s_i: nearest existing firehouse time
nearest_existing = [argmin(t_exist[:, i]) for i in eachindex(I_areas)]
status_quo = sum(w .* s) / sum(w)

@printf("t_ji (candidates): %d × %d, range %.1f–%.1f min\n", size(t)..., minimum(t), maximum(t))
@printf("t_ki (existing):   %d × %d, range %.1f–%.1f min\n", size(t_exist)..., minimum(t_exist), maximum(t_exist))
@printf("s_i: range %.1f–%.1f min;  status-quo call-weighted average: %.2f min\n", minimum(s), maximum(s), status_quo)
@printf("ZIPs where at least one candidate beats the nearest firehouse: %d of %d\n",
        count(i -> minimum(t[:, i]) < s[i], eachindex(I_areas)), length(I_areas))

t_ji (candidates): 20 × 37, range 1.5–52.0 min
t_ki (existing):   66 × 37, range 0.9–49.0 min
s_i: range 0.9–6.6 min;  status-quo call-weighted average: 2.16 min
ZIPs where at least one candidate beats the nearest firehouse: 5 of 37


#### 5. Build and solve the MIP

In [6]:
function solve_standby_model(t, s, w, P; time_limit=TIME_LIMIT_SEC, silent=true)
    nJ, nI = size(t)
    model = Model(Gurobi.Optimizer)
    silent && set_silent(model)
    set_time_limit_sec(model, time_limit)

    @variable(model, y[1:nJ], Bin)          # candidate j opened
    @variable(model, x[1:nJ, 1:nI], Bin)    # candidate j serves area i
    @variable(model, z[1:nI], Bin)          # area i stays with its nearest existing firehouse

    @objective(model, Min,
        sum(w[i] * (s[i] * z[i] + sum(t[j, i] * x[j, i] for j in 1:nJ)) for i in 1:nI) / sum(w))

    @constraint(model, open_P, sum(y) == P)
    @constraint(model, assign[i in 1:nI], z[i] + sum(x[j, i] for j in 1:nJ) == 1)
    @constraint(model, link[j in 1:nJ, i in 1:nI], x[j, i] <= y[j])

    optimize!(model)
    is_solved_and_feasible(model) || error("Solve failed: $(termination_status(model))")

    selected = findall(>(0.5), value.(y))
    # assigned[i] = index of the new site serving area i, or 0 if it stays with an existing firehouse
    assigned = [something(findfirst(>(0.5), value.(x[:, i])), 0) for i in 1:nI]
    return (model=model, selected=selected, assigned=assigned, objective=objective_value(model))
end

sol = solve_standby_model(t, s, w, P)
println("Status: ", termination_status(sol.model), " | solve time: ",
        round(solve_time(sol.model), digits=2), " s")
@printf("Call-weighted average travel time: %.2f min (status quo %.2f min, saving %.1f s per call)\n",
        sol.objective, status_quo, 60 * (status_quo - sol.objective))
@printf("ZIPs switched to a new site: %d of %d\n", count(>(0), sol.assigned), length(I_areas))

Set parameter Username
Set parameter LicenseID to value 2857454
Academic license - for non-commercial use only - expires 2027-08-26
Status: OPTIMAL | solve time: 0.01 s
Call-weighted average travel time: 2.06 min (status quo 2.16 min, saving 5.6 s per call)
ZIPs switched to a new site: 5 of 37


#### 6. Results

In [7]:
selected_sites = candidates[sol.selected, :]
selected_sites.n_areas_served = [count(==(j), sol.assigned) for j in sol.selected]
selected_sites.calls_served   = [sum(w[sol.assigned .== j]) for j in sol.selected]
selected_sites.minutes_saved_per_call = [
    let served = sol.assigned .== j
        any(served) ? sum(w[served] .* (s[served] .- t[j, served])) / sum(w[served]) : 0.0
    end
    for j in sol.selected
]
selected_sites

Row,site_id,lat,lon,source,n_areas_served,calls_served,minutes_saved_per_call
,String,Float64,Float64,String15,Int64,Float64,Float64
1,2,40.6611,-73.8861,alternate,1,2037.0,1.10665
2,4,40.6217,-73.9494,original,1,4008.0,1.3789
3,7,40.7072,-73.9431,original,1,8664.0,0.853935
4,13,40.6625,-73.949,alternate,1,4271.0,0.0708849
5,14,40.6276,-73.9831,original,1,2877.0,1.22963


In [8]:
served_by_new = sol.assigned .> 0
assignment = DataFrame(
    area_id            = I_areas,
    weight             = w,
    served_by          = ifelse.(served_by_new, "new", "existing"),
    site_id            = [a > 0 ? J_sites[a] : K_sites[nearest_existing[i]] for (i, a) in enumerate(sol.assigned)],
    site_name          = [a > 0 ? "Pin $(J_sites[a])" : existing.name[nearest_existing[i]] for (i, a) in enumerate(sol.assigned)],
    minutes            = [a > 0 ? t[a, i] : s[i] for (i, a) in enumerate(sol.assigned)],
    status_quo_minutes = s,
)
assignment.minutes_saved = assignment.status_quo_minutes .- assignment.minutes
sort(assignment, :minutes, rev=true)

Row,area_id,weight,served_by,site_id,site_name,minutes,status_quo_minutes,minutes_saved
,String,Float64,String,String,String,Float64,Float64,Float64
1,11234,4959.0,existing,FH_063,Engine 321,6.6271,6.6271,0.0
2,11239,2037.0,new,2,Pin 2,5.54015,6.6468,1.10665
3,11230,3850.0,existing,FH_052,Engine 250,3.6207,3.6207,0.0
4,11226,8791.0,existing,FH_043,Engine 281/Ladder 147,3.0615,3.0615,0.0
5,11229,4294.0,existing,FH_063,Engine 321,3.0367,3.0367,0.0
6,11210,4008.0,new,4,Pin 4,2.826,4.2049,1.3789
7,11223,4053.0,existing,FH_057,Engine 254/Ladder 153,2.7327,2.7327,0.0
8,11236,6522.0,existing,FH_059,Battalion 58/Engine 257/Ladder 170,2.6553,2.6553,0.0
9,11218,3081.0,existing,FH_045,Battalion 48/Engine 240,2.5795,2.5795,0.0


In [9]:
# Score any set of new sites using the same objective as the optimization model:
# each ZIP gets the faster of its nearest existing firehouse and its nearest new site.
function score_layout(t, s, weights, selected)
    nearest = isempty(selected) ? copy(s) : min.(s, vec(minimum(t[selected, :]; dims=1)))
    average = sum(weights .* nearest) / sum(weights)
    return (average=average, worst_zip=maximum(nearest))
end

# Baseline 1: Start with the ZIPs with the most calls.
# For each of the top P ZIPs, pick its nearest unused candidate.
function busiest_first(t, weights, P)
    remaining = collect(axes(t, 1))
    selected = Int[]
    busiest_zips = sortperm(weights; rev=true)[1:P]

    for i in busiest_zips
        position = argmin(t[remaining, i])
        push!(selected, remaining[position])
        deleteat!(remaining, position)
    end

    return selected
end

# Baseline 2: Add one site at a time, starting from the existing network.
# At each step, pick the addition that gives the best borough-wide average.
function greedy_add(t, s, weights, P)
    remaining = collect(axes(t, 1))
    selected = Int[]
    nearest = copy(s)

    for _ in 1:P
        scores = [sum(weights .* min.(nearest, t[j, :])) for j in remaining]
        position = argmin(scores)
        site = remaining[position]

        push!(selected, site)
        nearest = min.(nearest, t[site, :])
        deleteat!(remaining, position)
    end

    return selected
end

layouts = [Int[], busiest_first(t, w, P), greedy_add(t, s, w, P), sol.selected]
scores = [score_layout(t, s, w, selected) for selected in layouts]

# Confirm that our evaluation reproduces the solver's objective.
@assert isapprox(scores[4].average, sol.objective; atol=1e-6)

comparison = DataFrame(
    method = ["Status quo (P = 0)", "Busiest-first", "Greedy", "Optimization"],
    avg_minutes = [round(sc.average; digits=2) for sc in scores],
    worst_zip_minutes = [round(sc.worst_zip; digits=2) for sc in scores],
    seconds_saved_vs_status_quo = [round(60 * (status_quo - sc.average); digits=1) for sc in scores],
    selected_sites = [join(J_sites[selected], ", ") for selected in layouts],
)

display(comparison)

saved = scores[3].average - scores[4].average
@printf("Optimization saves %.1f seconds per call vs. greedy (%.1f s vs. status quo).\n",
        60 * saved, 60 * (status_quo - scores[4].average))

Row,method,avg_minutes,worst_zip_minutes,seconds_saved_vs_status_quo,selected_sites
,String,Float64,Float64,Float64,String
1,Status quo (P = 0),2.16,6.65,0.0,
2,Busiest-first,2.14,6.63,0.8,"2, 3, 11, 1, 13"
3,Greedy,2.06,6.63,5.6,"7, 4, 14, 2, 13"
4,Optimization,2.06,6.63,5.6,"2, 4, 7, 13, 14"


Optimization saves 0.0 seconds per call vs. greedy (5.6 s vs. status quo).


#### 7. Save outputs

In [10]:
mkpath(OUTPUT_DIR)
CSV.write(joinpath(OUTPUT_DIR, "selected_sites_with_existing_P$(P).csv"), selected_sites)
CSV.write(joinpath(OUTPUT_DIR, "area_assignment_with_existing_P$(P).csv"), assignment)
CSV.write(joinpath(OUTPUT_DIR, "comparison_with_existing_P$(P).csv"), comparison)
println("Wrote results to $(OUTPUT_DIR)/")

Wrote results to results/
